In [52]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import recall_score, precision_score
from sklearn.model_selection import GridSearchCV

In [53]:
data = pd.read_csv("Customer churn kaggle data.csv")

In [54]:
data = data.drop('customerID', axis = 1)
data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors = "coerce")

In [55]:
x = data.drop('Churn', axis = 1)
y = data['Churn'].map({"Yes": 1, "No": 0})

In [56]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42, stratify = y)

In [57]:
num_cols = x_train.select_dtypes(include = ['int64', 'float64']).columns
cat_cols = x_train.select_dtypes(include = 'object').columns

In [58]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy = "mean")),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy = "most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [59]:
preprocessor = ColumnTransformer([
    ('nums', num_pipeline, num_cols),
    ('cats', cat_pipeline, cat_cols)
])

In [60]:
pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('model', LogisticRegression(max_iter = 1000, class_weight = 'balanced'))
])

In [61]:
param_grid = {
    'model__C': [0.01, 0.1, 1, 10],
    'model__penalty': ['l1', 'l2'],
    'model__solver': ['liblinear']
}

In [62]:
grid = GridSearchCV(pipeline, param_grid, cv = 5, scoring = "recall", n_jobs = -1)

In [63]:
grid.fit(x_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessing',
                                        ColumnTransformer(transformers=[('nums',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer()),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')),
                                                                        ('cats',
                                                                         Pipeline(steps=[('imputer',
                                                                                 

In [64]:
best_model = grid.best_estimator_

In [65]:
y_pred = best_model.predict(x_test)

In [66]:
print("Recall:", recall_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))

Recall: 0.7887700534759359
Precision: 0.5042735042735043


In [67]:
print(grid.best_params_)

{'model__C': 1, 'model__penalty': 'l1', 'model__solver': 'liblinear'}
